# Aquisição de Mapas LCZ — As Quatro Formas de Obter um Mapa LCZ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/01_map_acquisition.pt.ipynb)

**Objetivo de aprendizagem**: ao final deste notebook você saberá como obter um mapa de Zonas Climáticas Locais (LCZ) para qualquer lugar do mundo usando as quatro funções de aquisição do LCZ4py — o conjunto de dados global aberto, duas variantes regionais validadas (Europa, CONUS) e um job enviado pelo usuário ao LCZ Generator — além de como gerenciar o cache local em disco que torna downloads repetidos instantâneos.

## Por que um mapa LCZ padronizado?

O **esquema de Zonas Climáticas Locais (LCZ)** (Stewart & Oke, 2012, *Bulletin of the American Meteorological Society*) foi criado para resolver um problema crônico na climatologia urbana: estudos descreviam suas áreas de estudo com rótulos informais e inconsistentes — "urbano", "periurbano", "rural", "centro" — que significavam coisas diferentes em cidades diferentes e tornavam praticamente impossível comparar intensidades de Ilha de Calor Urbana (ICU) entre estudos. Stewart & Oke propuseram 17 tipos de zona padronizados (classes 1–10 para superfícies construídas/urbanas — de compacto de alta densidade a industrial baixo e extenso — e classes 11–17, rotuladas A–G, para coberturas naturais como árvores densas, vegetação rasteira, rocha exposta e água) definidos por propriedades de superfície mensuráveis: fator de visão do céu, razão de aspecto, fração de área construída, fração de superfície impermeável, altura dos elementos de rugosidade e classe de rugosidade do terreno. Dois locais na mesma classe LCZ, em qualquer parte do mundo, devem ter resposta térmica semelhante à radiação solar, o que é exatamente o que torna o esquema útil como uma "moeda comum" para estudos de ICU, planejamento urbano e modelagem energética de edificações.

Um **mapa** que atribui cada pixel de uma cidade (ou de um continente) a uma dessas 17 classes é o que transforma o esquema de teoria em uma camada espacial utilizável. Como mapas LCZ são trabalhosos de produzir do zero (exigem imagens de satélite, classificadores de aprendizado de máquina e validação por especialistas), a comunidade convergiu para um pequeno número de produtos de mapa abertos e versionados, que pesquisadores baixam em vez de reconstruir. O LCZ4py — a versão em Python do pacote R LCZ4r (Anjos et al., 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) — encapsula a aquisição de todos eles atrás de uma interface consistente, de modo que alternar entre uma visão global e um produto regional validado localmente é uma mudança de uma linha.

## Os quatro caminhos de aquisição deste notebook

1. **`lcz_get_map`** — o mapa LCZ **global** (Demuzere et al., 2022), um único mosaico mundial com resolução de ~100 m construído a partir de imagens Sentinel-2 no Google Earth Engine. Use como escolha padrão para qualquer cidade, em qualquer lugar: garante a mesma metodologia em todo lugar, o que é exatamente o que torna comparações entre cidades válidas.
2. **`lcz_get_map_euro`** — o mapa LCZ **europeu** (Demuzere et al., 2019). Um produto regional com validação e dados de treinamento adicionais específicos para cidades europeias; onde há sobreposição com o mapa global, espere pequenas diferenças nos limites das classes por causa dos dados regionais extras.
3. **`lcz_get_map_usa`** — o mapa LCZ **CONUS** (Demuzere et al., 2020), construído sobre o NLCD (National Land Cover Database) e cobrindo apenas os Estados Unidos continentais. Troca cobertura global por validação e dados auxiliares específicos dos EUA.
4. **`lcz_get_map_generator`** — baixa um **job específico, enviado pelo usuário**, da plataforma web [LCZ Generator](https://lcz-generator.rub.de/), identificado por um `id` de job. Este é um caminho de aquisição fundamentalmente diferente dos três primeiros: em vez de recortar um mosaico global/regional pré-existente para sua área de interesse, você está buscando a classificação personalizada de uma pessoa (seus próprios polígonos de treinamento, sua própria data de imagem), que pode cobrir uma área menor ou com formato diferente e vem com sua própria avaliação de acurácia (visível na página do factsheet).

As três funções baseadas em mosaico (`lcz_get_map`, `_euro`, `_usa`) compartilham a mesma interface: passe um nome de `city` (geocodificado automaticamente) ou um `roi` GeoDataFrame personalizado, e receba de volta o caminho para um GeoTIFF recortado. Por baixo dos panos, elas transmitem apenas os pixels que intersectam sua área de interesse a partir de um Cloud-Optimized GeoTIFF (COG) remoto — nenhum arquivo global de vários gigabytes chega a tocar seu disco — e armazenam o resultado em cache local, de modo que rodar a mesma cidade novamente é instantâneo.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. `lcz_get_map` — o mapa LCZ global

`lcz_get_map(city=None, roi=None, isave_map=False, cache=True, cache_dir="~/.lcz4r_cache", lang="en", verbose=True) -> str` é o ponto de entrada principal do LCZ4py. Passe um nome de `city` (geocodificado via Nominatim) ou um `roi` GeoDataFrame personalizado; a função retorna o **caminho para um GeoTIFF** recortado no limite dessa área, com valores inteiros de pixel 1–17 (classes LCZ) e 0 para nodata.

Parâmetros-chave:
- `isave_map`: também copia o raster recortado para `LCZ4r_output/` (caso contrário, ele fica apenas no diretório de cache).
- `cache` / `cache_dir`: com `cache=True` (padrão), tanto o limite geocodificado (GeoJSON/GeoArrow) quanto o raster recortado (GeoTIFF) são salvos em `cache_dir`, identificados por um hash do nome da cidade e da bounding box — rodar a mesma cidade novamente é quase instantâneo. Use `cache=False` para forçar um novo download em um diretório temporário.
- `verbose`: imprime o progresso (geocodificação, streaming, cache) no log.

Usamos **Vaduz** (a pequena capital de Liechtenstein) como exemplo condutor ao longo deste notebook: sua bounding box minúscula mantém cada download em poucos segundos, o que é ideal para aprender a API.

In [2]:
from LCZ4py import lcz_get_map
import rasterio
import numpy as np

map_path = lcz_get_map(city="Vaduz", verbose=True)
print("Clipped LCZ map saved at:", map_path)

with rasterio.open(map_path) as src:
    arr = src.read(1)
    print("shape:", arr.shape, "| CRS:", src.crs, "| bounds:", src.bounds)
    classes, counts = np.unique(arr, return_counts=True)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:28:28 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


Clipped LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif
shape: (120, 131) | CRS: EPSG:4326 | bounds: BoundingBox(left=9.494294237858497, bottom=47.0869922476931, right=9.611973540078154, top=47.19479008178744)
LCZ classes present (0 = nodata): {0: 13181, 5: 101, 6: 306, 8: 22, 9: 119, 11: 1260, 12: 88, 14: 244, 15: 389, 17: 10}


**Interpretando a saída**: `map_path` aponta para um GeoTIFF pequeno contendo apenas os pixels que intersectam o limite administrativo de Vaduz — tudo fora dele é nodata (0). O histograma de classes acima mostra, rapidamente, a composição da morfologia urbana da cidade: uma alta proporção de classes 1–10 (tipos construídos) versus 11–17 (tipos naturais/água) indica uma área mais urbanizada, enquanto uma paisagem dominada por classes como 11 (árvores densas) ou 15 (árvores esparsas) indica um entorno majoritariamente natural/rural. Esta é a entrada bruta sobre a qual todas as outras funções do LCZ4py nesta série se baseiam.

## 2. `lcz_clear_cache` — limpando o cache em disco

`lcz_clear_cache(cache_dir=None) -> int` apaga todo arquivo de limite em cache (`study_area_*`) e de raster recortado (`clipped_*`) do diretório de cache (padrão `~/.lcz4r_cache`, o mesmo padrão usado por `lcz_get_map`). Retorna o **número de arquivos apagados**. Use-a quando quiser forçar um novo download para todas as cidades (por exemplo, após uma atualização da fonte do mapa), ou simplesmente para liberar espaço em disco.

In [3]:
from LCZ4py import lcz_clear_cache

n_deleted = lcz_clear_cache()
print(f"Deleted {n_deleted} cached file(s).")

Deleted 7 cached file(s).


**Interpretando a saída**: a contagem reflete as entradas de cache de limite + raster removidas (aqui, as criadas pela chamada de Vaduz acima). As funções abaixo simplesmente vão repopular o cache sob demanda — limpá-lo nunca quebra nada, apenas troca espaço em disco por um próximo download um pouco mais lento.

## 3. `lcz_get_map_euro` — o mapa LCZ europeu

`lcz_get_map_euro(city=None, roi=None, isave_map=False, isave_euro=False, cache=True, cache_dir=..., lang="en", verbose=True) -> str` tem exatamente a mesma interface de `lcz_get_map`, além de um parâmetro `isave_euro` mantido apenas por paridade de API com o pacote R (a implementação em Python com streaming COG torna desnecessário salvar o mosaico europeu completo, então esse parâmetro não tem efeito). A fonte é o mapa LCZ europeu de **Demuzere et al. (2019)**, em vez do mosaico global. Use esta variante para cidades europeias quando quiser a validação regional extra embutida nesse produto, ao custo de cobrir apenas a Europa.

In [4]:
from LCZ4py import lcz_get_map_euro

map_path_euro = lcz_get_map_euro(city="Vaduz", verbose=True)
print("Clipped European LCZ map saved at:", map_path_euro)

with rasterio.open(map_path_euro) as src:
    arr_euro = src.read(1)
    classes, counts = np.unique(arr_euro, return_counts=True)
    print("shape:", arr_euro.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:28:30 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Vaduz' via async HTTP...


06:28:30 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Vaduz&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"


06:28:30 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_vaduz.arrow


06:28:30 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


06:28:37 - rasterio._env - WARNING - CPLE_IllegalArg in tmpuks7yqik.tif: BLOCKXSIZE can only be used with TILED=YES


06:28:37 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


Clipped European LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_8755df8f6d014dec.tif
shape: (80, 87) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 5830, 5: 2, 6: 181, 8: 3, 9: 36, 11: 586, 12: 151, 14: 88, 15: 3, 17: 80}


**Interpretando a saída**: compare este histograma de classes com o do mapa global acima. Pequenas diferenças nos limites de classe ou na atribuição de classe entre os dois são esperadas e refletem os dados extras de treinamento/validação específicos da Europa usados para construir este produto regional — nenhum dos dois está "errado", são simplesmente instantâneos diferentes da mesma realidade subjacente, com calibração local diferente.

## 4. `lcz_get_map_usa` — o mapa LCZ CONUS

`lcz_get_map_usa(city=None, roi=None, isave_map=False, isave_usa=False, cache=True, cache_dir=..., lang="en", verbose=True) -> str` espelha a mesma interface novamente, desta vez usando o mapa LCZ **CONUS (CONtinental United States)** de Demuzere et al. (2020), construído sobre dados de cobertura do solo do NLCD. Como seu mosaico raster cobre apenas os EUA continentais, **esta função exige uma cidade ou ROI dentro dos EUA** — diferentemente de `lcz_get_map`/`lcz_get_map_euro`, Vaduz (Liechtenstein) ficaria totalmente fora de sua extensão e geraria um erro. Trocamos o exemplo condutor para uma pequena cidade dos EUA, **Truckee, Califórnia**, para manter o download igualmente rápido.

In [5]:
from LCZ4py import lcz_get_map_usa

map_path_usa = lcz_get_map_usa(city="Truckee, California", verbose=True)
print("Clipped CONUS LCZ map saved at:", map_path_usa)

with rasterio.open(map_path_usa) as src:
    arr_usa = src.read(1)
    classes, counts = np.unique(arr_usa, return_counts=True)
    print("shape:", arr_usa.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:28:37 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Truckee, California' via async HTTP...


06:28:37 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Truckee%2C+California&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"


06:28:37 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_truckee,_california.arrow


06:28:37 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


06:28:40 - rasterio._env - WARNING - CPLE_IllegalArg in tmpy6jux_i5.tif: BLOCKXSIZE can only be used with TILED=YES


06:28:40 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


Clipped CONUS LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_05b9aad9021c0c57.tif
shape: (66, 208) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 6014, 6: 2259, 11: 600, 12: 4242, 13: 256, 14: 20, 16: 12, 17: 325}


**Interpretando a saída**: Truckee é uma pequena cidade de montanha, então espera-se que o histograma penda fortemente para classes naturais (por exemplo, 12/árvores densas, 13/árvores esparsas, 15/água ou rocha exposta em altitude), com apenas uma fatia fina de classes construídas baixas/abertas (5, 6, 8) perto do centro da cidade — uma checagem útil de que o produto CONUS está se comportando como esperado para um pequeno assentamento americano de baixa densidade.

## 5. `lcz_get_map_generator` — um job específico do LCZ Generator

`lcz_get_map_generator(id="3110e623fbe4e73b1cde55f0e9832c4f5640ac21", band="lczFilter", isave_map=False, save_extension="tif") -> str` é um tipo diferente de aquisição: em vez de recortar um mosaico global/regional para sua área, ela baixa **um job de classificação específico** enviado por um usuário à plataforma web [LCZ Generator](https://lcz-generator.rub.de/), identificado por seu `id` (visível na URL pública do factsheet do job). `band` seleciona qual camada de saída manter: `"lcz"` (saída bruta do classificador) ou `"lczFilter"` (padrão — o mapa pós-processado, filtrado/suavizado espacialmente, geralmente preferido). O `id` padrão usado abaixo é um job de exemplo público, então funciona sem que você precise ter enviado nada. Este caminho importa quando você precisa de uma classificação com uma avaliação de acurácia específica, data de treinamento ou polígonos de treinamento personalizados que nem o mosaico global nem um regional podem oferecer.

In [6]:
from LCZ4py import lcz_get_map_generator

map_path_gen = lcz_get_map_generator()  # default id: a public example job
print("LCZ Generator map saved at:", map_path_gen)

with rasterio.open(map_path_gen) as src:
    arr_gen = src.read(1)
    classes, counts = np.unique(arr_gen, return_counts=True)
    print("shape:", arr_gen.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:28:40 - LCZ4py.general.lcz_get_map_generator - INFO - Streaming map from LCZ Generator...


06:28:41 - httpx - INFO - HTTP Request: GET https://lcz-generator.rub.de/factsheets/3110e623fbe4e73b1cde55f0e9832c4f5640ac21/3110e623fbe4e73b1cde55f0e9832c4f5640ac21.tif "HTTP/1.1 200 OK"


LCZ Generator map saved at: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmp_udsi4fv.tif
shape: (727, 1435) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 2870, 1: 2275, 2: 767, 3: 61473, 4: 155, 5: 63, 6: 63777, 7: 1687, 8: 5710, 9: 44621, 10: 207, 11: 244735, 12: 43415, 13: 25230, 14: 214151, 15: 518, 16: 3191, 17: 328400}


**Interpretando a saída**: este raster tipicamente cobre uma extensão muito maior do que os pequenos recortes de Vaduz/Truckee acima, já que jobs do LCZ Generator costumam ser enviados para regiões metropolitanas inteiras. Note que, diferentemente das três funções de mosaico, não há parâmetro `city`/`roi` aqui — a extensão espacial é a que o autor original definiu para o job, não algo que você controla a partir do LCZ4py.

## Recapitulação e próximos passos

Agora você tem quatro formas de colocar um mapa LCZ em disco: `lcz_get_map` (padrão global), `lcz_get_map_euro` / `lcz_get_map_usa` (variantes regionais com validação local extra, ao custo de cobertura restrita), e `lcz_get_map_generator` (um job específico enviado por um usuário). As três funções de mosaico compartilham um cache em disco que você pode inspecionar e limpar com `lcz_clear_cache`.

Todo caminho de GeoTIFF retornado aqui (`map_path`, `map_path_euro`, `map_path_usa`, `map_path_gen`) é exatamente o tipo de entrada que o restante do conjunto de ferramentas de propósito geral do LCZ4py espera. **Próximo: `02_visualization_area_stats`** — visualizando esses mapas interativamente e calculando a área ocupada por cada classe LCZ.